# run_loc_pipeline.ipynb (Event-first)

Primary unit = **super-event** (recommended for sparse camera-trap). Outputs: `events_behavior.csv`, `behavior_summary_table.csv`, `summary.json`.


In [255]:
# Cell 0: Bootstrap so `import src...` works (robust)
import sys
from pathlib import Path

nb_dir = Path.cwd()
p = nb_dir.resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
if not (p / "src").exists():
    raise RuntimeError("Cannot find repo root containing 'src' directory from current working dir.")

repo_root = p
sys.path.insert(0, str(repo_root))

print("Notebook dir:", nb_dir)
print("Repo root:", repo_root)
print("sys.path[0]:", sys.path[0])


Notebook dir: c:\Users\78222\Desktop\Bicylist-System\bicyclist-system\notebooks
Repo root: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system
sys.path[0]: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system


In [256]:
# Cell 1: Imports
import re, json
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from src.scene.roi import load_roi_config, ROIMaskEngine
from src.inference.common import frame_space_label, FrameLabelThresholds


In [257]:
\
# Cell 2: User config (EDIT THIS)
DATA_ROOT = Path(r"C:\Users\78222\Desktop\28_locations\0_MAIN_BIKE_DATASETS_clean")

LOC_ID = "loc_25"  # e.g., "loc_04", "loc_04-2"
IMG_DIR = DATA_ROOT / "Loc_25" / "Bicyclist"

ROI_JSON = repo_root / "configs" / "locations" / f"{LOC_ID}.json"
OUTDIR = repo_root / "outputs" / f"{LOC_ID}_eventA"
OUTDIR.mkdir(parents=True, exist_ok=True)

EVENT_GAP = 2
SUPER_GAP = 7

USE_YOLO = True
YOLO_MODEL = "yolov8n.pt"
TARGET_CLASS_IDS = {1}  # bicycle
CONF_MIN = 0.10
MAX_IMAGES = None  # e.g., 200 for debug

BOTTOM_EDGE_NPTS = 5
VOTE_MIN_PTS = 2
MIN_MOVE_PX = 8.0

print("LOC_ID:", LOC_ID)
print("IMG_DIR:", IMG_DIR)
print("ROI_JSON:", ROI_JSON)
print("OUTDIR:", OUTDIR)
assert IMG_DIR.exists(), f"Missing IMG_DIR: {IMG_DIR}"
assert ROI_JSON.exists(), f"Missing ROI_JSON: {ROI_JSON}"


LOC_ID: loc_25
IMG_DIR: C:\Users\78222\Desktop\28_locations\0_MAIN_BIKE_DATASETS_clean\Loc_25\Bicyclist
ROI_JSON: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system\configs\locations\loc_25.json
OUTDIR: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system\outputs\loc_25_eventA


In [258]:
# Cell 3: Collect images
img_paths = sorted(list(IMG_DIR.glob("*.JPG")))
if not img_paths:
    img_paths = sorted(list(IMG_DIR.glob("*.jpg")))
assert len(img_paths) > 0, f"No images found in {IMG_DIR}"

if MAX_IMAGES is not None:
    img_paths = img_paths[:MAX_IMAGES]

print("num images:", len(img_paths))
print("first/last:", img_paths[0].name, img_paths[-1].name)

img0 = Image.open(img_paths[0])
print("real image_size:", img0.size)


num images: 9
first/last: IM_16386.JPG IM_19053.JPG
real image_size: (1920, 1088)


In [259]:
# Cell 4: Burst events + Super-events
def extract_img_number(name: str) -> int:
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else -1

def group_images_into_events(img_paths, gap=2):
    imgs = sorted(img_paths, key=lambda p: extract_img_number(p.name))
    if len(imgs) == 0:
        return []
    events = []
    cur = [imgs[0]]
    for prev, now in zip(imgs, imgs[1:]):
        if (extract_img_number(now.name) - extract_img_number(prev.name)) <= gap:
            cur.append(now)
        else:
            events.append(cur)
            cur = [now]
    events.append(cur)
    return events

def merge_events_into_super_events(events, super_gap=7):
    if len(events) == 0:
        return []
    def num(p: Path) -> int:
        return extract_img_number(p.name)
    super_events = []
    cur = list(events[0])
    for prev_ev, next_ev in zip(events, events[1:]):
        prev_last = num(prev_ev[-1])
        next_first = num(next_ev[0])
        if (next_first - prev_last) <= super_gap:
            cur.extend(next_ev)
        else:
            super_events.append(cur)
            cur = list(next_ev)
    super_events.append(cur)
    return super_events

events = group_images_into_events(img_paths, gap=EVENT_GAP)
super_events = merge_events_into_super_events(events, super_gap=SUPER_GAP)

print("num burst events:", len(events))
print("num super-events:", len(super_events))


num burst events: 9
num super-events: 9


In [260]:
# Cell 5: YOLO detections -> detections_df (wide)
detections_df = pd.DataFrame(columns=["img","frame_global","x1","y1","x2","y2","score","cls"])

if USE_YOLO:
    from ultralytics import YOLO
    model = YOLO(YOLO_MODEL)

    rows = []
    for i, p in enumerate(img_paths):
        r = model(str(p), verbose=False)[0]
        if r.boxes is None:
            continue
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clss  = r.boxes.cls.cpu().numpy().astype(int)

        for (x1,y1,x2,y2), s, c in zip(boxes, confs, clss):
            if c not in TARGET_CLASS_IDS:
                continue
            if float(s) < CONF_MIN:
                continue
            rows.append({
                "img": p.name,
                "frame_global": int(i),
                "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
                "score": float(s),
                "cls": int(c),
            })
    detections_df = pd.DataFrame(rows)

print("detections_df:", detections_df.shape)
display(detections_df.head(10))
detections_df.to_csv(OUTDIR / "detections_raw.csv", index=False)


detections_df: (4, 8)


,img,frame_global,x1,y1,x2,y2,score,cls
0,IM_16386.JPG,0,243.306152,778.603638,617.799805,1026.548584,0.558303,1
1,IM_17967.JPG,5,212.341827,672.339233,325.597504,910.027710,0.698779,1
2,IM_18369.JPG,6,105.398621,822.034546,541.771362,1030.191162,0.385223,1
3,IM_18369.JPG,6,227.432510,823.742981,457.832642,1025.179199,0.255148,1


In [261]:
# Cell 6: Frame-level ROI labeling (bottom-edge voting)
cfg = load_roi_config(ROI_JSON)
roi = ROIMaskEngine(cfg)
flow = roi.flow_vector()

has_crosswalk = bool(cfg.rois.get("crosswalk")) and len(cfg.rois.get("crosswalk", [])) > 0
print("has_crosswalk:", has_crosswalk)
print("flow_vector:", flow)

thr = FrameLabelThresholds(bottom_edge_npts=BOTTOM_EDGE_NPTS, vote_min_pts=VOTE_MIN_PTS)

def _label_row(r):
    bb = (float(r.x1), float(r.y1), float(r.x2), float(r.y2))
    return frame_space_label(bb, roi, thr)

d = detections_df.copy()
d["space_label"] = d.apply(_label_row, axis=1)

KEEP_LABELS = ["sidewalk", "bike_lane", "roadway"]
if has_crosswalk:
    KEEP_LABELS.append("crosswalk")

d = d[d["space_label"].isin(KEEP_LABELS)].copy()

print("detections in study ROIs:", len(d))
print(d["space_label"].value_counts(dropna=False))

d.to_csv(OUTDIR / "detections_labeled.csv", index=False)


has_crosswalk: False
flow_vector: (0.9769632573303413, 0.21340757677854263)
detections in study ROIs: 1
space_label
bike_lane    1
Name: count, dtype: int64


In [262]:
# Cell 7: Super-event aggregation (occupancy + wrong-way)
img2idx = {p.name: i for i, p in enumerate(img_paths)}

def bbox_bottom_center(bb):
    x1,y1,x2,y2 = bb
    return ((x1+x2)/2.0, y2)

def summarize_one_event(ev_imgs):
    ev_names = [p.name if isinstance(p, Path) else str(p) for p in ev_imgs]
    det_ev = d[d["img"].isin(ev_names)].copy()
    if len(det_ev) == 0:
        return None

    c_side = int((det_ev["space_label"]=="sidewalk").sum())
    c_bike = int((det_ev["space_label"]=="bike_lane").sum())
    c_road = int((det_ev["space_label"]=="roadway").sum())
    c_cw   = int((det_ev["space_label"]=="crosswalk").sum()) if has_crosswalk else 0
    total  = c_side + c_bike + c_road + c_cw

    in_side = c_side > 0
    in_bike = c_bike > 0
    in_road = (c_road + c_cw) > 0

    dom = "unknown"
    if total > 0:
        dom = max([("sidewalk",c_side),("bike_lane",c_bike),("roadway",c_road+c_cw)], key=lambda x:x[1])[0]

    wrong_way = pd.NA
    direction = pd.NA
    cos_to_flow = pd.NA

    if flow is not None:
        det_ev["frame_global"] = det_ev["img"].map(img2idx)
        det_ev = det_ev.dropna(subset=["frame_global"])

        if len(det_ev) >= 2:
            fg0 = int(det_ev["frame_global"].min())
            fg1 = int(det_ev["frame_global"].max())

            det0 = det_ev[det_ev["frame_global"] == fg0].sort_values("score", ascending=False).head(1)
            det1 = det_ev[det_ev["frame_global"] == fg1].sort_values("score", ascending=False).head(1)

            if len(det0) == 1 and len(det1) == 1:
                bb0 = (float(det0.iloc[0].x1), float(det0.iloc[0].y1), float(det0.iloc[0].x2), float(det0.iloc[0].y2))
                bb1 = (float(det1.iloc[0].x1), float(det1.iloc[0].y1), float(det1.iloc[0].x2), float(det1.iloc[0].y2))
                bc0 = bbox_bottom_center(bb0)
                bc1 = bbox_bottom_center(bb1)

                vx, vy = bc1[0]-bc0[0], bc1[1]-bc0[1]
                mag = float((vx*vx + vy*vy)**0.5)

                if mag >= MIN_MOVE_PX:
                    ux, uy = vx/mag, vy/mag
                    cos = float(ux*flow[0] + uy*flow[1])
                    cos_to_flow = cos
                    direction = "along_flow" if cos >= 0 else "against_flow"
                    wrong_way = bool(cos < 0)

    return {
        "n_imgs": len(ev_imgs),
        "has_target": True,
        "n_det": int(total),
        "in_sidewalk_any": bool(in_side),
        "in_bike_lane_any": bool(in_bike),
        "in_roadway_any": bool(in_road),
        "dominant_space": dom,
        "wrong_way": wrong_way,
        "direction": direction,
        "cos_to_flow": cos_to_flow,
    }

rows = []
for eid, ev in enumerate(super_events):
    r = summarize_one_event(ev)
    if r is None:
        rows.append({"event_id": eid, "n_imgs": len(ev), "has_target": False})
    else:
        r["event_id"] = eid
        rows.append(r)

events_behavior_df = pd.DataFrame(rows)
print("events_behavior_df:", events_behavior_df.shape)
display(events_behavior_df.head(10))
events_behavior_df.to_csv(OUTDIR / "events_behavior.csv", index=False)


events_behavior_df: (9, 11)


,event_id,n_imgs,has_target,n_det,in_sidewalk_any,in_bike_lane_any,in_roadway_any,dominant_space,wrong_way,direction,cos_to_flow
0,0,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,1,True,1.0,False,True,False,bike_lane,<NA>,<NA>,<NA>
6,6,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [263]:
# Cell 8: Build behavior summary table (occupancy + wrong-way; crossing omitted if no crosswalk)
df = pd.read_csv(OUTDIR / "events_behavior.csv")
df_has = df[df["has_target"] == True].copy()
N = len(df_has)
print("events with target:", N)

rows = []

for k in ["in_sidewalk_any","in_bike_lane_any","in_roadway_any"]:
    rows.append({
        "behavior": k,
        "count": int((df_has[k] == True).sum()),
        "denominator": N,
        "ratio": round(float((df_has[k] == True).mean()), 3) if N else None
    })

app = df_has["wrong_way"].notna()
n_app = int(app.sum())
n_ww = int((df_has.loc[app, "wrong_way"] == True).sum()) if n_app else 0
rows.append({
    "behavior": "wrong_way",
    "count": n_ww,
    "denominator": n_app,
    "ratio": round((n_ww / n_app), 3) if n_app else None
})

vc = df_has["dominant_space"].value_counts(dropna=False)
for k, v in vc.items():
    rows.append({
        "behavior": f"dominant_space={k}",
        "count": int(v),
        "denominator": N,
        "ratio": round(float(v / N), 3) if N else None
    })

behavior_summary = pd.DataFrame(rows)
display(behavior_summary)
behavior_summary.to_csv(OUTDIR / "behavior_summary_table.csv", index=False)

summary = {
    "location_id": LOC_ID,
    "unit": "super_event",
    "N_super_events": int(len(df)),
    "N_has_target": int(N),
    "N_wrong_way_applicable": n_app,
    "N_wrong_way": n_ww,
    "share_wrong_way": (n_ww / n_app) if n_app else None,
    "params": {
        "EVENT_GAP": EVENT_GAP,
        "SUPER_GAP": SUPER_GAP,
        "CONF_MIN": CONF_MIN,
        "BOTTOM_EDGE_NPTS": BOTTOM_EDGE_NPTS,
        "VOTE_MIN_PTS": VOTE_MIN_PTS,
        "MIN_MOVE_PX": MIN_MOVE_PX,
    }
}
(OUTDIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", OUTDIR / "behavior_summary_table.csv")
print("Saved:", OUTDIR / "summary.json")
print(json.dumps(summary, indent=2))


events with target: 1


,behavior,count,denominator,ratio
0,in_sidewalk_any,0,1,0.0
1,in_bike_lane_any,1,1,1.0
2,in_roadway_any,0,1,0.0
3,wrong_way,0,0,NaN
4,dominant_space=bike_lane,1,1,1.0


Saved: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system\outputs\loc_25_eventA\behavior_summary_table.csv
Saved: C:\Users\78222\Desktop\Bicylist-System\bicyclist-system\outputs\loc_25_eventA\summary.json
{
  "location_id": "loc_25",
  "unit": "super_event",
  "N_super_events": 9,
  "N_has_target": 1,
  "N_wrong_way_applicable": 0,
  "N_wrong_way": 0,
  "share_wrong_way": null,
  "params": {
    "EVENT_GAP": 2,
    "SUPER_GAP": 7,
    "CONF_MIN": 0.1,
    "BOTTOM_EDGE_NPTS": 5,
    "VOTE_MIN_PTS": 2,
    "MIN_MOVE_PX": 8.0
  }
}
